# Module 7: Generative Models

This notebook implements the core generative model architectures.

**Topics covered:**
- Variational Autoencoders (VAE)
- Generative Adversarial Networks (GAN)
- Diffusion Models (DDPM)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

np.random.seed(42)
%matplotlib inline

## 7.1 Variational Autoencoder (VAE)

In [ ]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class VAE:
    """
    Variational Autoencoder.
    
    Learns a latent distribution q(z|x) and decoder p(x|z).
    Uses the reparameterization trick for training.
    """
    
    def __init__(self, input_dim, hidden_dim, latent_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        scale = 0.1
        
        # Encoder
        self.W_enc1 = np.random.randn(input_dim, hidden_dim) * scale
        self.b_enc1 = np.zeros(hidden_dim)
        
        # Encoder outputs mean and log-variance
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * scale
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * scale
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder
        self.W_dec1 = np.random.randn(latent_dim, hidden_dim) * scale
        self.b_dec1 = np.zeros(hidden_dim)
        self.W_dec2 = np.random.randn(hidden_dim, input_dim) * scale
        self.b_dec2 = np.zeros(input_dim)
    
    def encode(self, x):
        """Encode input to latent distribution parameters."""
        h = relu(x @ self.W_enc1 + self.b_enc1)
        mu = h @ self.W_mu + self.b_mu
        logvar = h @ self.W_logvar + self.b_logvar
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick.
        
        Instead of sampling z ~ N(mu, sigma^2),
        sample eps ~ N(0, 1) and compute z = mu + sigma * eps.
        This allows gradients to flow through.
        """
        std = np.exp(0.5 * logvar)
        eps = np.random.randn(*mu.shape)
        return mu + std * eps
    
    def decode(self, z):
        """Decode latent vector to reconstruction."""
        h = relu(z @ self.W_dec1 + self.b_dec1)
        return sigmoid(h @ self.W_dec2 + self.b_dec2)
    
    def forward(self, x):
        """Full forward pass."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar, z
    
    def loss(self, x, x_recon, mu, logvar):
        """
        ELBO loss = Reconstruction + KL divergence.
        
        Reconstruction: -E[log p(x|z)] ≈ BCE(x, x_recon)
        KL: KL(q(z|x) || p(z)) where p(z) = N(0, I)
           = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
        """
        # Binary cross-entropy reconstruction loss
        recon_loss = -np.mean(
            x * np.log(x_recon + 1e-10) + 
            (1 - x) * np.log(1 - x_recon + 1e-10)
        )
        
        # KL divergence (closed form for Gaussians)
        kl_loss = -0.5 * np.mean(
            1 + logvar - mu**2 - np.exp(logvar)
        )
        
        return recon_loss + kl_loss, recon_loss, kl_loss
    
    def sample(self, n_samples):
        """Sample from the prior and decode."""
        z = np.random.randn(n_samples, self.latent_dim)
        return self.decode(z)

In [ ]:
# Create synthetic 2D data (mixture of Gaussians)
def create_2d_data(n_samples=500):
    """Create 2D data from mixture of Gaussians."""
    centers = [(0, 0), (3, 3), (3, -3), (-3, 3), (-3, -3)]
    data = []
    for _ in range(n_samples):
        center = centers[np.random.randint(len(centers))]
        point = np.random.randn(2) * 0.5 + center
        data.append(point)
    return np.array(data)

# Normalize to [0, 1] for sigmoid output
data = create_2d_data(1000)
data_min = data.min(axis=0)
data_max = data.max(axis=0)
data_normalized = (data - data_min) / (data_max - data_min + 1e-10)

plt.figure(figsize=(8, 6))
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, s=10)
plt.title('Training Data: Mixture of Gaussians')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Create and test VAE
vae = VAE(input_dim=2, hidden_dim=64, latent_dim=2)

# Forward pass
x = data_normalized[:10]
x_recon, mu, logvar, z = vae.forward(x)

print("VAE Forward Pass:")
print(f"  Input shape: {x.shape}")
print(f"  Latent mu shape: {mu.shape}")
print(f"  Latent logvar shape: {logvar.shape}")
print(f"  Sampled z shape: {z.shape}")
print(f"  Reconstruction shape: {x_recon.shape}")

# Compute loss
total_loss, recon_loss, kl_loss = vae.loss(x, x_recon, mu, logvar)
print(f"\nLoss breakdown:")
print(f"  Reconstruction: {recon_loss:.4f}")
print(f"  KL divergence: {kl_loss:.4f}")
print(f"  Total (ELBO): {total_loss:.4f}")

In [ ]:
# Visualize the VAE components
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original data
axes[0, 0].scatter(data_normalized[:, 0], data_normalized[:, 1], alpha=0.3, s=10)
axes[0, 0].set_title('Original Data (normalized)')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')

# Latent space encoding
all_mu, all_logvar = vae.encode(data_normalized)
axes[0, 1].scatter(all_mu[:, 0], all_mu[:, 1], alpha=0.3, s=10)
axes[0, 1].set_title('Encoded Latent Space (mu)')
axes[0, 1].set_xlabel('z1')
axes[0, 1].set_ylabel('z2')

# Samples from prior
prior_samples = np.random.randn(500, 2)
decoded_samples = vae.decode(prior_samples)
axes[1, 0].scatter(decoded_samples[:, 0], decoded_samples[:, 1], alpha=0.3, s=10)
axes[1, 0].set_title('Decoded Samples from Prior')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')

# Reconstructions
recon, _, _, _ = vae.forward(data_normalized)
axes[1, 1].scatter(recon[:, 0], recon[:, 1], alpha=0.3, s=10)
axes[1, 1].set_title('Reconstructions')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('y')

plt.tight_layout()
plt.show()

## 7.2 Generative Adversarial Network (GAN)

In [ ]:
class Generator:
    """
    GAN Generator: Maps noise to data space.
    G(z) -> x_fake
    """
    
    def __init__(self, latent_dim, hidden_dim, output_dim):
        scale = 0.1
        self.W1 = np.random.randn(latent_dim, hidden_dim) * scale
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * scale
        self.b2 = np.zeros(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, output_dim) * scale
        self.b3 = np.zeros(output_dim)
    
    def forward(self, z):
        """Generate fake samples from noise."""
        h1 = relu(z @ self.W1 + self.b1)
        h2 = relu(h1 @ self.W2 + self.b2)
        out = h2 @ self.W3 + self.b3  # Linear output (tanh in practice)
        return out

class Discriminator:
    """
    GAN Discriminator: Classifies real vs fake.
    D(x) -> probability of being real
    """
    
    def __init__(self, input_dim, hidden_dim):
        scale = 0.1
        self.W1 = np.random.randn(input_dim, hidden_dim) * scale
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * scale
        self.b2 = np.zeros(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, 1) * scale
        self.b3 = np.zeros(1)
    
    def forward(self, x):
        """Output probability that x is real."""
        h1 = relu(x @ self.W1 + self.b1)
        h2 = relu(h1 @ self.W2 + self.b2)
        logit = h2 @ self.W3 + self.b3
        return sigmoid(logit)

In [ ]:
# Create GAN
latent_dim = 2
hidden_dim = 64
data_dim = 2

G = Generator(latent_dim, hidden_dim, data_dim)
D = Discriminator(data_dim, hidden_dim)

# Test forward passes
z = np.random.randn(10, latent_dim)
fake_samples = G.forward(z)
real_samples = data[:10]

print("GAN Components:")
print(f"  Noise z shape: {z.shape}")
print(f"  Generated fake shape: {fake_samples.shape}")
print(f"  D(real) probabilities: {D.forward(real_samples).flatten()[:5]}")
print(f"  D(fake) probabilities: {D.forward(fake_samples).flatten()[:5]}")

In [ ]:
def gan_discriminator_loss(D, real_data, fake_data):
    """
    Discriminator loss (maximize):
    L_D = E[log D(x)] + E[log(1 - D(G(z)))]
    
    We want D(real) -> 1 and D(fake) -> 0
    """
    d_real = D.forward(real_data)
    d_fake = D.forward(fake_data)
    
    loss = -np.mean(np.log(d_real + 1e-10) + np.log(1 - d_fake + 1e-10))
    return loss

def gan_generator_loss(D, fake_data):
    """
    Generator loss (minimize, non-saturating version):
    L_G = -E[log D(G(z))]
    
    We want D(fake) -> 1 (fool discriminator)
    """
    d_fake = D.forward(fake_data)
    loss = -np.mean(np.log(d_fake + 1e-10))
    return loss

# Compute losses
z = np.random.randn(100, latent_dim)
fake = G.forward(z)
real = data[:100]

d_loss = gan_discriminator_loss(D, real, fake)
g_loss = gan_generator_loss(D, fake)

print(f"Discriminator loss: {d_loss:.4f}")
print(f"Generator loss: {g_loss:.4f}")

In [ ]:
# Visualize GAN training dynamics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Real data
axes[0].scatter(data[:, 0], data[:, 1], alpha=0.3, s=10, label='Real')
axes[0].set_title('Real Data')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

# Generated (untrained)
z = np.random.randn(500, latent_dim)
fake = G.forward(z)
axes[1].scatter(fake[:, 0], fake[:, 1], alpha=0.3, s=10, c='orange', label='Fake')
axes[1].set_title('Generated (Untrained)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()

# Discriminator decision boundary
xx, yy = np.meshgrid(np.linspace(-5, 5, 50), np.linspace(-5, 5, 50))
grid = np.c_[xx.ravel(), yy.ravel()]
d_scores = D.forward(grid).reshape(xx.shape)

axes[2].contourf(xx, yy, d_scores, levels=20, cmap='RdYlGn', alpha=0.7)
axes[2].scatter(data[:200, 0], data[:200, 1], alpha=0.5, s=10, c='blue', label='Real')
axes[2].scatter(fake[:200, 0], fake[:200, 1], alpha=0.5, s=10, c='red', label='Fake')
axes[2].set_title('Discriminator Scores\n(Green=Real, Red=Fake)')
axes[2].colorbar(axes[2].collections[0], label='D(x)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 7.3 Diffusion Models (DDPM)

In [ ]:
class SimpleDiffusion:
    """
    Simple Diffusion Model (DDPM concepts).
    
    Forward process: q(x_t | x_0) = N(sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)
    
    The model learns to predict the noise added at each step.
    """
    
    def __init__(self, num_timesteps=100, beta_start=0.0001, beta_end=0.02):
        self.num_timesteps = num_timesteps
        
        # Linear noise schedule
        self.betas = np.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1 - self.betas
        self.alpha_bars = np.cumprod(self.alphas)
    
    def q_sample(self, x_0, t, noise=None):
        """
        Sample from q(x_t | x_0).
        
        x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
        """
        if noise is None:
            noise = np.random.randn(*x_0.shape)
        
        sqrt_alpha_bar = np.sqrt(self.alpha_bars[t])
        sqrt_one_minus_alpha_bar = np.sqrt(1 - self.alpha_bars[t])
        
        # Handle broadcasting for batch
        if x_0.ndim > 1:
            sqrt_alpha_bar = sqrt_alpha_bar.reshape(-1, 1)
            sqrt_one_minus_alpha_bar = sqrt_one_minus_alpha_bar.reshape(-1, 1)
        
        return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise, noise
    
    def get_snr(self, t):
        """
        Signal-to-noise ratio at timestep t.
        SNR = alpha_bar / (1 - alpha_bar)
        """
        return self.alpha_bars[t] / (1 - self.alpha_bars[t])

In [ ]:
# Visualize the forward diffusion process
diffusion = SimpleDiffusion(num_timesteps=100)

# Create a simple 2D point
x_0 = np.array([[2.0, 2.0]])

# Show noising at different timesteps
timesteps = [0, 20, 40, 60, 80, 99]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, t in enumerate(timesteps):
    # Sample many noisy versions
    x_0_batch = np.tile(x_0, (200, 1))
    t_batch = np.array([t] * 200)
    x_t, _ = diffusion.q_sample(x_0_batch, t_batch)
    
    axes[idx].scatter(x_t[:, 0], x_t[:, 1], alpha=0.3, s=10)
    axes[idx].scatter(x_0[0, 0], x_0[0, 1], c='red', s=100, marker='x', label='Original')
    axes[idx].set_xlim(-4, 4)
    axes[idx].set_ylim(-4, 4)
    axes[idx].set_title(f't={t}, ᾱ={diffusion.alpha_bars[t]:.3f}')
    axes[idx].set_aspect('equal')
    axes[idx].grid(True, alpha=0.3)
    if idx == 0:
        axes[idx].legend()

plt.suptitle('Forward Diffusion Process: Progressive Noising', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the noise schedule
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Beta schedule
axes[0].plot(diffusion.betas)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('β_t')
axes[0].set_title('Beta Schedule (Noise Added per Step)')
axes[0].grid(True, alpha=0.3)

# Alpha bar schedule
axes[1].plot(diffusion.alpha_bars)
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('ᾱ_t')
axes[1].set_title('Alpha Bar (Signal Remaining)')
axes[1].grid(True, alpha=0.3)

# SNR
snr = [diffusion.get_snr(t) for t in range(diffusion.num_timesteps)]
axes[2].semilogy(snr)
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('SNR (log scale)')
axes[2].set_title('Signal-to-Noise Ratio')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
class NoisePredictor:
    """
    Simple MLP to predict the noise added at timestep t.
    
    In practice, this would be a U-Net with time embeddings.
    """
    
    def __init__(self, data_dim, hidden_dim, num_timesteps):
        scale = 0.1
        # Time embedding
        self.time_embed = np.random.randn(num_timesteps, hidden_dim) * scale
        
        # Network
        self.W1 = np.random.randn(data_dim + hidden_dim, hidden_dim) * scale
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * scale
        self.b2 = np.zeros(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, data_dim) * scale
        self.b3 = np.zeros(data_dim)
    
    def forward(self, x_t, t):
        """Predict noise given noisy input and timestep."""
        # Get time embedding
        t_emb = self.time_embed[t]
        if t_emb.ndim == 1:
            t_emb = t_emb.reshape(1, -1)
        if t_emb.shape[0] == 1 and x_t.shape[0] > 1:
            t_emb = np.tile(t_emb, (x_t.shape[0], 1))
        
        # Concatenate input with time embedding
        x = np.concatenate([x_t, t_emb], axis=-1)
        
        # Forward through network
        h1 = relu(x @ self.W1 + self.b1)
        h2 = relu(h1 @ self.W2 + self.b2)
        eps_pred = h2 @ self.W3 + self.b3
        
        return eps_pred

def diffusion_loss(model, diffusion, x_0, t):
    """
    Diffusion training loss.
    
    L = E[||eps - eps_theta(x_t, t)||^2]
    """
    # Sample noise and create noisy input
    x_t, noise = diffusion.q_sample(x_0, t)
    
    # Predict noise
    noise_pred = model.forward(x_t, t)
    
    # MSE loss
    loss = np.mean((noise - noise_pred) ** 2)
    return loss

In [ ]:
# Test the noise predictor
noise_model = NoisePredictor(data_dim=2, hidden_dim=64, num_timesteps=100)

# Sample some data
x_0 = data[:32]
t = np.random.randint(0, 100, 32)

loss = diffusion_loss(noise_model, diffusion, x_0, t)
print(f"Diffusion loss (untrained): {loss:.4f}")

In [ ]:
def ddpm_sample_step(model, diffusion, x_t, t):
    """
    One step of DDPM reverse sampling.
    
    p(x_{t-1} | x_t) = N(mu_theta, sigma^2 I)
    
    where mu_theta = (1/sqrt(alpha_t)) * (x_t - (beta_t/sqrt(1-alpha_bar_t)) * eps_theta)
    """
    if t == 0:
        return x_t
    
    # Get schedule values
    alpha_t = diffusion.alphas[t]
    alpha_bar_t = diffusion.alpha_bars[t]
    beta_t = diffusion.betas[t]
    
    # Predict noise
    t_batch = np.array([t] * x_t.shape[0])
    eps_pred = model.forward(x_t, t_batch)
    
    # Compute mean
    mu = (1 / np.sqrt(alpha_t)) * (
        x_t - (beta_t / np.sqrt(1 - alpha_bar_t)) * eps_pred
    )
    
    # Add noise (except at t=0)
    sigma = np.sqrt(beta_t)
    noise = np.random.randn(*x_t.shape)
    
    return mu + sigma * noise

def ddpm_sample(model, diffusion, n_samples, dim):
    """Full DDPM sampling from pure noise."""
    # Start from pure noise
    x = np.random.randn(n_samples, dim)
    
    # Reverse diffusion
    trajectory = [x.copy()]
    for t in reversed(range(diffusion.num_timesteps)):
        x = ddpm_sample_step(model, diffusion, x, t)
        if t % 20 == 0:  # Save every 20 steps
            trajectory.append(x.copy())
    
    return x, trajectory

In [ ]:
# Visualize reverse sampling (with untrained model)
samples, trajectory = ddpm_sample(noise_model, diffusion, n_samples=200, dim=2)

fig, axes = plt.subplots(1, len(trajectory), figsize=(18, 3))

for idx, traj in enumerate(trajectory):
    axes[idx].scatter(traj[:, 0], traj[:, 1], alpha=0.3, s=10)
    axes[idx].set_xlim(-4, 4)
    axes[idx].set_ylim(-4, 4)
    if idx == 0:
        axes[idx].set_title(f't={diffusion.num_timesteps-1}\n(noise)')
    elif idx == len(trajectory) - 1:
        axes[idx].set_title(f't=0\n(sample)')
    else:
        t = diffusion.num_timesteps - 1 - (idx * 20)
        axes[idx].set_title(f't≈{t}')
    axes[idx].set_aspect('equal')

plt.suptitle('DDPM Reverse Process (Untrained - Random Walk)', fontsize=14)
plt.tight_layout()
plt.show()

## 7.4 Comparing Generative Models

In [ ]:
# Comparison table visualization
comparison_data = {
    'Model': ['VAE', 'GAN', 'Diffusion'],
    'Training': ['Stable', 'Tricky', 'Stable'],
    'Sample Quality': ['Good', 'Excellent', 'Excellent'],
    'Diversity': ['High', 'Mode collapse risk', 'High'],
    'Speed': ['Fast', 'Fast', 'Slow'],
    'Likelihood': ['Tractable (ELBO)', 'Intractable', 'Tractable']
}

print("Generative Model Comparison:")
print("=" * 80)
print(f"{'Property':<20} {'VAE':<20} {'GAN':<20} {'Diffusion':<20}")
print("-" * 80)

for prop in ['Training', 'Sample Quality', 'Diversity', 'Speed', 'Likelihood']:
    idx_vae = comparison_data['Model'].index('VAE')
    idx_gan = comparison_data['Model'].index('GAN')
    idx_diff = comparison_data['Model'].index('Diffusion')
    
    values = comparison_data[prop]
    print(f"{prop:<20} {values[0]:<20} {values[1]:<20} {values[2]:<20}")

print("=" * 80)

In [ ]:
# Visualize the different generation approaches
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# VAE: Sample from learned latent space
z_vae = np.random.randn(500, 2)
samples_vae = vae.decode(z_vae)
axes[0].scatter(samples_vae[:, 0], samples_vae[:, 1], alpha=0.3, s=10)
axes[0].set_title('VAE Samples\n(Decode from N(0,I))')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# GAN: Transform noise through generator
z_gan = np.random.randn(500, 2)
samples_gan = G.forward(z_gan)
axes[1].scatter(samples_gan[:, 0], samples_gan[:, 1], alpha=0.3, s=10, c='orange')
axes[1].set_title('GAN Samples\n(Generator from N(0,I))')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')

# Diffusion: Iterative denoising
samples_diff, _ = ddpm_sample(noise_model, diffusion, 500, 2)
axes[2].scatter(samples_diff[:, 0], samples_diff[:, 1], alpha=0.3, s=10, c='green')
axes[2].set_title('Diffusion Samples\n(Iterative denoising)')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')

plt.suptitle('Generative Model Samples (All Untrained)', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we covered:

1. **VAE**: Learns latent distribution via ELBO = Reconstruction + KL
   - Uses reparameterization trick for training
   - Samples from learned prior

2. **GAN**: Adversarial training between Generator and Discriminator
   - G tries to fool D, D tries to distinguish real/fake
   - Can produce high-quality samples but training is unstable

3. **Diffusion**: Forward noising + learned reverse denoising
   - Train noise predictor on q(x_t | x_0)
   - Generate by iteratively denoising from pure noise
   - Slow but high quality and diverse

**Next:** Module 8 covers Advanced Topics (RAG, Agents, Evaluation).